# AGH DS Laboratory 6 - Actor Model with Ray Framework

## Introduction

Ray is a general-purpose framework for programming a cluster made by UC Berkeley's RISELab. It enables developers to easily parallelize their Python applications or build new ones, and run them at any scale, from a laptop to a large cluster. It also provides a highly flexible, yet minimalist and easy to use API. 

#### Documentation Reference Links:

Ray documentation: https://docs.ray.io/en/latest/ \
Ray GitHub: https://github.com/ray-project/ray \
Ray Core: https://docs.ray.io/en/latest/ray-core/walkthrough.html \
Ray Client: https://docs.ray.io/en/latest/cluster/running-applications/job-submission/ray-client.html \

***
## Part 1 - Remote Functions

This script is too slow, and the computation is embarrassingly parallel. In this exercise, you will use Ray to execute the functions in parallel to speed it up.

The standard way to turn a Python function into a remote function is to add the `@ray.remote` decorator. Here is an example.

```python
# A regular Python function.
def regular_function(x):
    return x + 1

# A Ray remote function.
@ray.remote
def remote_function(x):
    return x + 1
```

The differences are the following:

1. **Invocation:** The regular version is called with `regular_function(1)`, whereas the remote version is called with `remote_function.remote(1)`.
2. **Return values:** `regular_function` immediately executes and returns `1`, whereas `remote_function` immediately returns an object ID (a future) and then creates a task that will be executed on a worker process. The result can be obtained with `ray.get`.
    ```python
    >>> regular_function(0)
    1
    
    >>> remote_function.remote(0)
    ObjectID(1c80d6937802cd7786ad25e50caf2f023c95e350)
    
    >>> ray.get(remote_function.remote(0))
    1
    ```
3. **Parallelism:** Invocations of `regular_function` happen **serially**, for example
    ```python
    # These happen serially.
    for _ in range(4):
        regular_function(0)
    ```
    whereas invocations of `remote_function` happen in **parallel**, for example
    ```python
    # These happen in parallel.
    for _ in range(4):
        remote_function.remote(0)
    ```

In [31]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import ray
import time
import numpy as np
from numpy import random
import os
import pickle

import sys

print("Python:", sys.version)
print("Ray:", ray.__version__)

Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:56) [GCC 14.3.0]
Ray: 2.55.1


Start Ray. By default, Ray does not schedule more tasks concurrently than there are CPUs. This example requires four tasks to run concurrently, so we tell Ray that there are four CPUs. Usually this is not done and Ray computes the number of CPUs using `psutil.cpu_count()`. The argument `ignore_reinit_error=True` just ignores errors if the cell is run multiple times.

The call to `ray.init` starts a number of processes.

In [32]:
STUDENT_ID = "naziemiec_blazej"  # WPISZ SWOJE IMIE I NAZWISKO
RAY_ADDRESS = "ray://ac8f571eeeaf941d38972ffb3fae2503-1174959016.us-east-1.elb.amazonaws.com:10001"

if ray.is_initialized():
    ray.shutdown()

ray.init(
    # address=RAY_ADDRESS,
    ignore_reinit_error=True,
    #namespace=f"lab-ray-{STUDENT_ID}",
)

print("Connected")
print("Student:", STUDENT_ID)
print("Namespace:", ray.get_runtime_context().namespace)
print("Job ID:", ray.get_runtime_context().get_job_id())
print(ray.cluster_resources())


2026-05-31 20:17:48,096	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 2147483648 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=2.45gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-05-31 20:17:50,216	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Connected
Student: naziemiec_blazej
Namespace: f39765c5-d8dc-4bf6-8a3e-ec603412ad05
Job ID: 01000000
{'CPU': 10.0, 'object_store_memory': 2395715174.0, 'node:172.18.0.3': 1.0, 'node:__internal_head__': 1.0, 'memory': 5590002074.0}


Create a helloWorld Actor and verify it shows on the dashboard.                      

In [33]:
ACTOR_NAME = f"actor-{STUDENT_ID}"

@ray.remote
class Hello(object):
    def __init__(self, x):
        self.x = x
    
    def set(self, x):
        self.x = x
    
    def get(self):
        return self.x

hello = Hello.options(name=ACTOR_NAME).remote(10)


In [34]:
# This function is a proxy for a more interesting and computationally
# intensive function.

@ray.remote
def slow_function(i):
    print(f"Running in pid {os.getpid()}")
    time.sleep(1)
    return i

**EXERCISE:** The loop below takes too long. The four function calls could be executed in parallel. Instead of four seconds, it should only take one second. Once `slow_function` has been made a remote function, execute these four tasks in parallel by calling `slow_function.remote()`. Then obtain the results by calling `ray.get` on a list of the resulting object IDs.

In [35]:
# Sleep a little to improve the accuracy of the timing measurements below.
# We do this because workers may still be starting up in the background.
time.sleep(2.0)
start_time = time.time()

results_ref = [slow_function.remote(i) for i in range(4)]
results = ray.get(results_ref)

end_time = time.time()
duration = end_time - start_time

print('The results are {}. This took {} seconds. Run the next cell to see '
      'if the exercise was done correctly.'.format(results, duration))

(slow_function pid=2962) Running in pid 2962
The results are [0, 1, 2, 3]. This took 1.116788387298584 seconds. Run the next cell to see if the exercise was done correctly.


**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [36]:
assert results == [0, 1, 2, 3], 'Did you remember to call ray.get?'
assert duration < 3.1, ('The loop took {} seconds. This is too slow.'
                        .format(duration))
assert duration > 1, ('The loop took {} seconds. This is too fast.'
                      .format(duration))

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 1.116788387298584 seconds.


***Task: Batch Processing***
Let's implement a simple batch image/data processing simulation. In the cell below, you have a function that processes many independent “log chunks”. Each chunk takes about 1 second.

In [37]:
import time
import random
from collections import Counter
import socket

EVENT_TYPES = ["login", "logout", "purchase", "click", "error"]


def process_log_chunk(chunk_id, chunk_size=10_000):
    """
    Simulate processing one chunk of logs.
    """
    time.sleep(1)  # simulate I/O or expensive parsing

    events = [
        random.choice(EVENT_TYPES)
        for _ in range(chunk_size)
    ]

    counts = Counter(events)

    return {
        "chunk_id": chunk_id,
        "counts": dict(counts),
        #"hostname": socket.gethostname(), #uncomment the line for remote version
    }

Now execute it sequentially

In [38]:
start = time.time()

sequential_results = [
    process_log_chunk(i)
    for i in range(8)
]

sequential_time = time.time() - start

print("Sequential time:", sequential_time)
sequential_results[:2]

Sequential time: 8.078516006469727


[{'chunk_id': 0,
  'counts': {'purchase': 2065,
   'login': 1965,
   'click': 2004,
   'error': 1953,
   'logout': 2013}},
 {'chunk_id': 1,
  'counts': {'purchase': 2007,
   'click': 2013,
   'error': 1958,
   'logout': 1980,
   'login': 2042}}]

Next implement task that will execute the processing logic on distributed cluster (name task as *process_log_chunk_remote*)

In [39]:
@ray.remote
def process_log_chunk_remote(chunk_id, chunk_size=10_000):
    time.sleep(1)
    events = [
        random.choice(EVENT_TYPES)
        for _ in range(chunk_size)
    ]
    counts = Counter(events)
    return {
        "chunk_id": chunk_id,
        "counts": dict(counts),
        #"hostname": socket.gethostname(),
    }

Execute the task in parallel - show the time difference in processing compared to the sequential version.

In [40]:
start = time.time()

parallel_refs = [
    process_log_chunk_remote.remote(i)
    for i in range(8)
]
parallel_results = ray.get(parallel_refs)

parallel_time = time.time() - start
print("Parallel time:", parallel_time)
print("Speedup:", sequential_time / parallel_time)
parallel_results[:2]

Parallel time: 1.1296110153198242
Speedup: 7.151591031699062


[{'chunk_id': 0,
  'counts': {'purchase': 1996,
   'logout': 1962,
   'login': 2015,
   'click': 2045,
   'error': 1982}},
 {'chunk_id': 1,
  'counts': {'logout': 1905,
   'click': 2082,
   'purchase': 1978,
   'login': 2039,
   'error': 1996}}]

As example of an anti-pattern, execute tasks in sequential manner using the ray framework, print processing time

In [41]:
start = time.time()

antipattern_results = []
for i in range(8):
    antipattern_results.append(ray.get(process_log_chunk_remote.remote(i)))

antipattern_time = time.time() - start
print("Antipattern time:", antipattern_time)

Antipattern time: 8.13550329208374


Next, compute the total counts of logs on your local Python environment. Keep in mind that this time you do not store anything on the cluster - everything remains on your local notebook environment. It is not the Actor model because you execute everything on the local machine.

In [42]:
total_counts = Counter()
for res in parallel_results:
    total_counts.update(res["counts"])

print("Total counts of logs:", dict(total_counts))

Total counts of logs: {'purchase': 15909, 'logout': 15954, 'login': 16074, 'click': 16089, 'error': 15974}


Finally measure the time for different number of iteration 4,8,16,32 
* When does adding more tasks stop improving runtime?
* How many CPUs does Ray report?                       

In [43]:
print("Total Ray CPUs:", ray.cluster_resources().get("CPU", 0))
print("Free Ray CPUs:", ray.available_resources().get("CPU", 0))

for num_tasks in [4, 8, 16, 32]:
    start = time.time()
    refs = [process_log_chunk_remote.remote(i) for i in range(num_tasks)]
    ray.get(refs)
    duration = time.time() - start
    print(f"Time for {num_tasks} tasks: {duration:.4f} seconds")

Total Ray CPUs: 10.0
Free Ray CPUs: 9.0
Time for 4 tasks: 1.0164 seconds
Time for 8 tasks: 1.0224 seconds
Time for 16 tasks: 2.0357 seconds
Time for 32 tasks: 4.0706 seconds


***
## Part 2 - Parallel Data Processing with Task Dependencies

**GOAL:** The goal of this exercise is to show how to pass object IDs into remote functions to encode dependencies between tasks.

In this exercise, we construct a sequence of tasks each of which depends on the previous mimicking a data parallel application. Within each sequence, tasks are executed serially, but multiple sequences can be executed in parallel.

In this exercise, you will use Ray to parallelize the computation below and speed it up.

### Concept for this Exercise - Task Dependencies

Suppose we have two remote functions defined as follows.

```python
@ray.remote
def f(x):
    return x
```

Arguments can be passed into remote functions as usual.

```python
>>> x1_id = f.remote(1)
>>> ray.get(x1_id)
1

>>> x2_id = f.remote([1, 2, 3])
>>> ray.get(x2_id)
[1, 2, 3]
```

**Object IDs** can also be passed into remote functions. When the function actually gets executed, **the argument will be a retrieved as a regular Python object**.

```python
>>> y1_id = f.remote(x1_id)
>>> ray.get(y1_id)
1

>>> y2_id = f.remote(x2_id)
>>> ray.get(y2_id)
[1, 2, 3]
```

So when implementing a remote function, the function should expect a regular Python object regardless of whether the caller passes in a regular Python object or an object ID.

**Task dependencies affect scheduling.** In the example above, the task that creates `y1_id` depends on the task that creates `x1_id`. This has the following implications.

- The second task will not be executed until the first task has finished executing.
- If the two tasks are scheduled on different machines, the output of the first task (the value corresponding to `x1_id`) will be copied over the network to the machine where the second task is scheduled.


These are some helper functions that mimic an example pattern of a data parallel application.

**EXERCISE:** You will need to turn all of these functions into remote functions. When you turn these functions into remote function, you do not have to worry about whether the caller passes in an object ID or a regular object. In both cases, the arguments will be regular objects when the function executes. This means that even if you pass in an object ID, you **do not need to call `ray.get`** inside of these remote functions.

In [44]:
@ray.remote
def load_data(filename):
    time.sleep(0.1)
    return np.ones((1000, 100))

@ray.remote
def normalize_data(data):
    time.sleep(0.1)
    return data - np.mean(data, axis=0)

@ray.remote
def extract_features(normalized_data):
    time.sleep(0.1)
    return np.hstack([normalized_data, normalized_data ** 2])

@ray.remote
def compute_loss(features):
    num_data, dim = features.shape
    time.sleep(0.1)
    return np.sum((np.dot(features, np.ones(dim)) - np.ones(num_data)) ** 2)

assert hasattr(load_data, 'remote'), 'load_data must be a remote function'
assert hasattr(normalize_data, 'remote'), 'normalize_data must be a remote function'
assert hasattr(extract_features, 'remote'), 'extract_features must be a remote function'
assert hasattr(compute_loss, 'remote'), 'compute_loss must be a remote function'

**EXERCISE:** The loop below takes too long. Parallelize the four passes through the loop by turning `load_data`, `normalize_data`, `extract_features`, and `compute_loss` into remote functions and then retrieving the losses with `ray.get`.

**NOTE:** You should only use **ONE** call to `ray.get`. For example, the object ID returned by `load_data` should be passed directly into `normalize_data` without needing to be retrieved by the driver.

In [45]:
# Sleep a little to improve the accuracy of the timing measurements below.
time.sleep(2.0)
start_time = time.time()

losses = []
for filename in ['file1', 'file2', 'file3', 'file4']:
    inner_start = time.time()

    data = load_data.remote(filename)
    normalized_data = normalize_data.remote(data)
    features = extract_features.remote(normalized_data)
    loss = compute_loss.remote(features)
    losses.append(loss)
    
    inner_end = time.time()
    
    if inner_end - inner_start >= 0.5:
        raise Exception('You may be calling ray.get inside of the for loop! '
                        'Doing this will prevent parallelism from being exposed. '
                        'Make sure to only call ray.get once outside of the for loop.')

losses = ray.get(losses)
print('The losses are {}.'.format(losses) + '\n')
loss = sum(losses)

end_time = time.time()
duration = end_time - start_time

print('The loss is {}. This took {} seconds. Run the next cell to see '
      'if the exercise was done correctly.'.format(loss, duration))

The losses are [1000.0, 1000.0, 1000.0, 1000.0].

The loss is 4000.0. This took 0.45711660385131836 seconds. Run the next cell to see if the exercise was done correctly.


**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [46]:
assert loss == 4000
assert duration < 3.5, ('The loop took {} seconds. This is too slow.'
                        .format(duration))
assert duration > 0.4, ('The loop took {} seconds. This is too fast.'
                        .format(duration))

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 0.45711660385131836 seconds.


**Note:** If your tests fail and values are closed to reference it is OK (the cluster is slow, and there are communication costs)

***
## Part 3 - Introducing Actors

**Goal:** The goal of this exercise is to show how to create an actor and how to call actor methods.

See the documentation on actors at https://docs.ray.io/en/latest/ray-core/actors.html.

Sometimes you need a "worker" process to have "state". For example, that state might be a neural network, a simulator environment, a counter, or something else entirely. However, remote functions are side-effect free. That is, they operate on inputs and produce outputs, but they don't change the state of the worker they execute on.

Actors are different. When we instantiate an actor, a brand new worker is created, and all methods that are called on that actor are executed on the newly created worker.

This means that with a single actor, no parallelism can be achieved because calls to the actor's methods will be executed one at a time. However, multiple actors can be created and methods can be executed on them in parallel.

### Concepts for this Exercise - Actors

To create an actor, decorate Python class with the `@ray.remote` decorator.

```python
@ray.remote
class Example(object):
    def __init__(self, x):
        self.x = x
    
    def set(self, x):
        self.x = x
    
    def get(self):
        return self.x
```

Like regular Python classes, **actors encapsulate state that is shared across actor method invocations**.

Actor classes differ from regular Python classes in the following ways.
1. **Instantiation:** A regular class would be instantiated via `e = Example(1)`. Actors are instantiated via
    ```python
    e = Example.remote(1)
    ```
    When an actor is instantiated, a **new worker process** is created by a local scheduler somewhere in the cluster.
2. **Method Invocation:** Methods of a regular class would be invoked via `e.set(2)` or `e.get()`. Actor methods are invoked differently.
    ```python
    >>> e.set.remote(2)
    ObjectID(d966aa9b6486331dc2257522734a69ff603e5a1c)
    
    >>> e.get.remote()
    ObjectID(7c432c085864ed4c7c18cf112377a608676afbc3)
    ```
3. **Return Values:** Actor methods are non-blocking. They immediately return an object ID and **they create a task which is scheduled on the actor worker**. The result can be retrieved with `ray.get`.
    ```python
    >>> ray.get(e.set.remote(2))
    None
    
    >>> ray.get(e.get.remote())
    2
    ```

**EXERCISE:** Change the `Foo` class to be an actor class by using the `@ray.remote` decorator.

In [47]:
@ray.remote
class Foo(object):
    def __init__(self):
        self.counter = 0

    def reset(self):
        self.counter = 0

    def increment(self):
        time.sleep(0.5)
        self.counter += 1
        return self.counter

assert hasattr(Foo, 'remote'), 'You need to turn "Foo" into an actor with @ray.remote.'

**EXERCISE:** Change the intantiations below to create two actors by calling `Foo.remote()`.

In [48]:
# Create two Foo objects.
f1 = Foo.remote()
f2 = Foo.remote()

**EXERCISE:** Parallelize the code below. The two actors can execute methods in parallel (though each actor can only execute one method at a time).

In [49]:
time.sleep(2.0)
start_time = time.time()

f1.reset.remote()
f2.reset.remote()

results_refs = []
for _ in range(5):
    results_refs.append(f1.increment.remote())
    results_refs.append(f2.increment.remote())

results = ray.get(results_refs)

end_time = time.time()
duration = end_time - start_time

print('Success! The example took {} seconds.'.format(duration))

assert not any([isinstance(result, ray.ObjectID) for result in results]), 'Looks like "results" is {}. You may have forgotten to call ray.get.'.format(results)

Success! The example took 2.626905679702759 seconds.


**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [50]:
assert results == [1, 1, 2, 2, 3, 3, 4, 4, 5, 5]

assert duration < 3, ('The experiments ran in {} seconds. This is too '
                      'slow.'.format(duration))
assert duration > 2.5, ('The experiments ran in {} seconds. This is too '
                        'fast.'.format(duration))

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 2.626905679702759 seconds.


**LogAggregator Task** Next, create an actor that will aggregate log events from the *Task Exercise* part of the class. The actor should implement methods such as: 
1. add_chunk_result - adding chunk
2. get_count - total counts
3. get_processed_chunks - current processed chunks
4. get_summary - return num_chunks, num_events and total counts
5. reset - clears all store data


In [51]:
@ray.remote
class LogAggregator(object):
    def __init__(self):
        self.reset()

    def add_chunk_result(self, chunk):
        self.processed_chunks.append(chunk["chunk_id"])
        for event_type, count in chunk["counts"].items():
            self.total_counts[event_type] = self.total_counts.get(event_type, 0) + count
            self.num_events += count
        self.num_chunks += 1

    def get_count(self):
        return self.total_counts

    def get_processed_chunks(self):
        return self.processed_chunks

    def get_summary(self):
        return {
            "num_chunks": self.num_chunks,
            "num_events": self.num_events,
            "total_counts": self.total_counts
        }

    def reset(self):
        self.processed_chunks = []
        self.total_counts = {}
        self.num_events = 0
        self.num_chunks = 0

It will be named actor it means its name will be unique

In [52]:
ACTOR_NAME = f"actor-{STUDENT_ID}"

try:
    stale_aggregator = ray.get_actor(ACTOR_NAME)
    
    ray.kill(stale_aggregator)
except ValueError:
    print(f"No existing actor found for '{ACTOR_NAME}'.")

aggregator = LogAggregator.options(name=ACTOR_NAME).remote()
print(f"Created fresh actor with latest code: {ACTOR_NAME}")

ray.get(aggregator.reset.remote())

Created fresh actor with latest code: actor-naziemiec_blazej


Next collect all data, and display summary

In [53]:
time.sleep(2.0)

chunk_refs = [process_log_chunk_remote.remote(i) for i in range(8)]
add_refs = [aggregator.add_chunk_result.remote(ref) for ref in chunk_refs]

ray.get(add_refs)

summary = ray.get(aggregator.get_summary.remote())
print("Aggregator Summary:")
print(f"Processed chunks: {ray.get(aggregator.get_processed_chunks.remote())}")
print(f"Number of chunks: {summary['num_chunks']}")
print(f"Number of events: {summary['num_events']}")
print(f"Total counts: {summary['total_counts']}")

Aggregator Summary:
Processed chunks: [0, 1, 2, 3, 4, 5, 6, 7]
Number of chunks: 8
Number of events: 80000
Total counts: {'error': 16240, 'click': 15925, 'login': 16028, 'logout': 15953, 'purchase': 15854}


Next run more tasks and prove that the state is updated

In [54]:
more_chunk_refs = [process_log_chunk_remote.remote(i) for i in range(8, 12)]
more_add_refs = [aggregator.add_chunk_result.remote(ref) for ref in more_chunk_refs]

ray.get(more_add_refs)

updated_summary = ray.get(aggregator.get_summary.remote())
print("Updated Aggregator Summary:")
print(f"Processed chunks: {ray.get(aggregator.get_processed_chunks.remote())}")
print(f"Number of chunks: {updated_summary['num_chunks']}")
print(f"Number of events: {updated_summary['num_events']}")
print(f"Total counts: {updated_summary['total_counts']}")

Updated Aggregator Summary:
Processed chunks: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Number of chunks: 12
Number of events: 120000
Total counts: {'error': 24319, 'click': 23813, 'login': 24023, 'logout': 23934, 'purchase': 23911}


## Part 4 - More advanced Actors, queuing operations

**GOAL:** The goal of this exercise is to illustrate how to actors queues of different operations

### Concepts for this Exercise - learn different ways for serializing calls on actor 


As the base for the exercise, create the Actor that will implement bank account functionality. 
1. It should keep information regarding the owner, balance, and all operations.
2. The account should support deposit and withdrawal operations (both should have a 1-second sleep time for processing). On successful operations return set with owner, name of operation, amount, balance and hostname (socket.gethostname())
3. Add getters for  balance and history


In [55]:
@ray.remote
class BankAccount:
    def __init__(self, owner, initial_balance=1000):
        self.owner = owner
        self.balance = initial_balance
        self.history = []

    def deposit(self, amount):
        time.sleep(1)
        self.balance += amount
        result = {
            'owner': self.owner,
            'operation': 'deposit',
            'amount': amount,
            'balance': self.balance,
            'hostname': socket.gethostname()
        }
        self.history.append(result)
        return result

    def withdraw(self, amount):
        time.sleep(1)
        if amount > self.balance:
            raise ValueError(f'Insufficient funds: balance={self.balance}, requested={amount}')
        self.balance -= amount
        result = {
            'owner': self.owner,
            'operation': 'withdraw',
            'amount': amount,
            'balance': self.balance,
            'hostname': socket.gethostname()
        }
        self.history.append(result)
        return result

    def get_balance(self):
        return self.balance

    def get_history(self):
        return self.history

The following operations should pass

In [56]:
account = BankAccount.remote("kowalski", initial_balance=100)

start = time.time()

refs = [
    account.deposit.remote(10),
    account.deposit.remote(20),
    account.withdraw.remote(50),
    account.deposit.remote(5),
]

results = ray.get(refs)
duration = time.time() - start

print("Duration:", duration)
print("Results:")
for r in results:
    print(r)

print("Final balance:", ray.get(account.get_balance.remote()))
print("History:", ray.get(account.get_history.remote()))

Duration: 4.254556655883789
Results:
{'owner': 'kowalski', 'operation': 'deposit', 'amount': 10, 'balance': 110, 'hostname': 'e0a726a74829'}
{'owner': 'kowalski', 'operation': 'deposit', 'amount': 20, 'balance': 130, 'hostname': 'e0a726a74829'}
{'owner': 'kowalski', 'operation': 'withdraw', 'amount': 50, 'balance': 80, 'hostname': 'e0a726a74829'}
{'owner': 'kowalski', 'operation': 'deposit', 'amount': 5, 'balance': 85, 'hostname': 'e0a726a74829'}
Final balance: 85
History: [{'owner': 'kowalski', 'operation': 'deposit', 'amount': 10, 'balance': 110, 'hostname': 'e0a726a74829'}, {'owner': 'kowalski', 'operation': 'deposit', 'amount': 20, 'balance': 130, 'hostname': 'e0a726a74829'}, {'owner': 'kowalski', 'operation': 'withdraw', 'amount': 50, 'balance': 80, 'hostname': 'e0a726a74829'}, {'owner': 'kowalski', 'operation': 'deposit', 'amount': 5, 'balance': 85, 'hostname': 'e0a726a74829'}]


1. How long does it take to execute the four operations?
2. Why does it take about 4 seconds, even though the calls were submitted almost at the same time?
3. Is the final account balance correct?
4. Does the actor behave more like a function or like a process with a message queue?


1. To trwa około 4 sekund, ponieważ każdy z tych procesów aktorów jest wywoływany sekwencyjnie. Ponieważ każdy aktor współdzieli ten sam zasób (history/balance) to działania wynokują się synchronicznie
2. Ponieważ są wywoływanie aktorzy synchronicznie żeby zapobiedz `race condition` (każde wywołanie ma 1s oczekiwanie na początku, stąd bieze się lekko poand 4s)
3. Tak, balance jest odpowiedni, bo: 100 + 10 + 20 - 50 + 5 = 85
4. Bardziej zachowuje się jak proces z kolejką wiadomości, którą sekwencyjnie przetwarza jedno po drugim

Next create 4 different accounts and call single method on one of them, measure time

In [57]:
accounts = [
    BankAccount.remote(f'owner_{i}', initial_balance=500)
    for i in range(4)
]

start = time.time()

refs = [acc.deposit.remote(100) for acc in accounts]
results = ray.get(refs)

duration = time.time() - start
print('Duration:', duration)
for r in results:
    print(r)

Duration: 1.3455803394317627
{'owner': 'owner_0', 'operation': 'deposit', 'amount': 100, 'balance': 600, 'hostname': 'e0a726a74829'}
{'owner': 'owner_1', 'operation': 'deposit', 'amount': 100, 'balance': 600, 'hostname': 'e0a726a74829'}
{'owner': 'owner_2', 'operation': 'deposit', 'amount': 100, 'balance': 600, 'hostname': 'e0a726a74829'}
{'owner': 'owner_3', 'operation': 'deposit', 'amount': 100, 'balance': 600, 'hostname': 'e0a726a74829'}


1. Why does it now take about 1 second instead of 4?
2. What is the unit of parallelism: the method or the actor?
3. What is the relationship between the number of actors and parallelism?


1. Ponieważ każdy z aktorów ma swój własny zasób, który nie jest współdzielony, więc na spokojnie można współbieżnie wykonywać działania na każdy z aktorów
2. Jednostką współbieżności jest aktor, ponieważ wszystkie zadania opierają się o współdzielone jego zasoby
3. Im więcej aktorów tym większa współbieżność

Finally lets create concurent version of the Bank account

In [58]:
account2 = BankAccount.options(max_concurrency=4).remote("nowak", initial_balance=100)

start = time.time()

refs = [
    account2.deposit.remote(10),
    account2.deposit.remote(20),
    account2.withdraw.remote(50),
    account2.deposit.remote(5),
]

results = ray.get(refs)
duration = time.time() - start

print("Duration:", duration)
for r in results:
    print(r)

print("Final balance:", ray.get(account2.get_balance.remote()))
print("History:", ray.get(account2.get_history.remote()))

Duration: 1.2802867889404297
{'owner': 'nowak', 'operation': 'deposit', 'amount': 10, 'balance': 110, 'hostname': 'e0a726a74829'}
{'owner': 'nowak', 'operation': 'deposit', 'amount': 20, 'balance': 130, 'hostname': 'e0a726a74829'}
{'owner': 'nowak', 'operation': 'withdraw', 'amount': 50, 'balance': 85, 'hostname': 'e0a726a74829'}
{'owner': 'nowak', 'operation': 'deposit', 'amount': 5, 'balance': 135, 'hostname': 'e0a726a74829'}
Final balance: 85
History: [{'owner': 'nowak', 'operation': 'deposit', 'amount': 10, 'balance': 110, 'hostname': 'e0a726a74829'}, {'owner': 'nowak', 'operation': 'deposit', 'amount': 20, 'balance': 130, 'hostname': 'e0a726a74829'}, {'owner': 'nowak', 'operation': 'deposit', 'amount': 5, 'balance': 135, 'hostname': 'e0a726a74829'}, {'owner': 'nowak', 'operation': 'withdraw', 'amount': 50, 'balance': 85, 'hostname': 'e0a726a74829'}]


1. Was the execution faster?
2. Is the final state always correct?
3. Why can concurrency inside an actor be dangerous?
4. When is it better to use multiple actors, and when is it better to use a single actor with `max_concurrency`?
(https://docs.ray.io/en/latest/ray-core/api/doc/ray.actor.ActorClass.options.html)

1. Tym razem wykonanie było szybsze dzięki użyciu `max_concurrency=4`
2. Nie zawsze będzie dobry wynik. Ponieważ jednocześnie różne instancje funkcji są edytować ten sam zasób to dochodzi do `race condition` i mogą być błędne wyniki
3. Kiedy mamy edytowalne dane w aktorze
4. Wiele aktorów jest lepszych w przypadku, gdzie edytujemy dane aktora w wielu wątkach, czy kiedy każdy nowa instancja aktora to niezależny byt. Jeden aktor z `max_concurrency` jest lepszy dla działań `I/O` i niewspółdzielonych zasób oraz kiedy wiele wątków ma jedynie np. odcztywać dane, a nie je edytować

***
## Part 5 - Handling Slow Tasks

**GOAL:** The goal of this exercise is to show how to use `ray.wait` to avoid waiting for slow tasks.

See the documentation for ray.wait at https://docs.ray.io/en/latest/ray-core/api/doc/ray.wait.html.

This script starts 6 tasks, each of which takes a random amount of time to complete. We'd like to process the results in two batches (each of size 3). Change the code so that instead of waiting for a fixed set of 3 tasks to finish, we make the first batch consist of the first 3 tasks that complete. The second batch should consist of the 3 remaining tasks. Do this exercise by using `ray.wait`.

### Concepts for this Exercise - ray.wait

After launching a number of tasks, you may want to know which ones have finished executing. This can be done with `ray.wait`. The function works as follows.

```python
ready_ids, remaining_ids = ray.wait(object_ids, num_returns=1, timeout=None)
```

**Arguments:**
- `object_ids`: This is a list of object IDs.
- `num_returns`: This is maximum number of object IDs to wait for. The default value is `1`.
- `timeout`: This is the maximum amount of time in milliseconds to wait for. So `ray.wait` will block until either `num_returns` objects are ready or until `timeout` milliseconds have passed.

**Return values:**
- `ready_ids`: This is a list of object IDs that are available in the object store.
- `remaining_ids`: This is a list of the IDs that were in `object_ids` but are not in `ready_ids`, so the IDs in `ready_ids` and `remaining_ids` together make up all the IDs in `object_ids`.

Define a remote function that takes a variable amount of time to run.

In [59]:
@ray.remote
def f(i):
    np.random.seed(5 + i)
    x = np.random.uniform(0, 4)
    time.sleep(x)
    return i, time.time()

**EXERCISE:** Using `ray.wait`, change the code below so that `initial_results` consists of the outputs of the first three tasks to complete instead of the first three tasks that were submitted.

In [60]:
time.sleep(2.0)
start_time = time.time()

result_ids = [f.remote(i) for i in range(6)]

initial_ids, remaining_ids = ray.wait(result_ids, num_returns=3)
initial_results = ray.get(initial_ids)

end_time = time.time()
duration = end_time - start_time

**EXERCISE:** Change the code below so that `remaining_results` consists of the outputs of the last three tasks to complete.

In [61]:
remaining_results = ray.get(remaining_ids)

**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [62]:
assert len(initial_results) == 3
assert len(remaining_results) == 3

initial_indices = [result[0] for result in initial_results]
initial_times = [result[1] for result in initial_results]
remaining_indices = [result[0] for result in remaining_results]
remaining_times = [result[1] for result in remaining_results]

assert set(initial_indices + remaining_indices) == set(range(6))

assert duration < 1.5, ('The initial batch of ten tasks was retrieved in '
                        '{} seconds. This is too slow.'.format(duration))

assert duration > 0.8, ('The initial batch of ten tasks was retrieved in '
                        '{} seconds. This is too fast.'.format(duration))

assert max(initial_times) < min(remaining_times)

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 0.9067220687866211 seconds.


## Part 6 - Speed up Serialization

**GOAL:** The goal of this exercise is to illustrate how to speed up serialization by using `ray.put`.

### Concepts for this Exercise - ray.put

Object IDs can be created in multiple ways.
- They are returned by remote function calls.
- They are returned by actor method calls.
- They are returned by `ray.put`.

When an object is passed to `ray.put`, the object is serialized using the Apache Arrow format (see https://arrow.apache.org/ for more information about Arrow) and copied into a shared memory object store. This object will then be available to other workers on the same machine via shared memory. If it is needed by workers on another machine, it will be shipped under the hood.

**When objects are passed into a remote function, Ray puts them in the object store under the hood.** That is, if `f` is a remote function, the code

```python
x = np.zeros(1000)
f.remote(x)
```

is essentially transformed under the hood to

```python
x = np.zeros(1000)
x_id = ray.put(x)
f.remote(x_id)
```

The call to `ray.put` copies the numpy array into the shared-memory object store, from where it can be read by all of the worker processes (without additional copying). However, if you do something like

```python
for i in range(10):
    f.remote(x)
```

then 10 copies of the array will be placed into the object store. This takes up more memory in the object store than is necessary, and it also takes time to copy the array into the object store over and over. This can be made more efficient by placing the array in the object store only once as follows.

```python
x_id = ray.put(x)
for i in range(10):
    f.remote(x_id)
```

In this exercise, you will speed up the code below and reduce the memory footprint by calling `ray.put` on the neural net weights before passing them into the remote functions.

**WARNING:** This exercise requires a lot of memory to run. If this notebook is running within a Docker container, then the docker container must be started with a large shared-memory file system. This can be done by starting the docker container with the `--shm-size` flag.

In [63]:
neural_net_weights = {'variable{}'.format(i): np.random.normal(size=1000000)
                      for i in range(50)}

**EXERCISE:** Compare the time required to serialize the neural net weights and copy them into the object store using Ray versus the time required to pickle and unpickle the weights. The big win should be with the time required for *deserialization*.

Note that when you call `ray.put`, in addition to serializing the object, we are copying it into shared memory where it can be efficiently accessed by other workers on the same machine.

**NOTE:** You don't actually have to do anything here other than run the cell below and read the output.

**NOTE:** Sometimes `ray.put` can be faster than `pickle.dumps`. This is because `ray.put` leverages multiple threads when serializing large objects. Note that this is not possible with `pickle`.

In [64]:
print('Ray - serializing')
%time x_id = ray.put(neural_net_weights)
print('\nRay - deserializing')
%time x_val = ray.get(x_id)

print('\npickle - serializing')
%time serialized = pickle.dumps(neural_net_weights)
print('\npickle - deserializing')
%time deserialized = pickle.loads(serialized)

Ray - serializing
CPU times: user 0 ns, sys: 589 ms, total: 589 ms
Wall time: 147 ms

Ray - deserializing
CPU times: user 0 ns, sys: 761 μs, total: 761 μs
Wall time: 488 μs

pickle - serializing
CPU times: user 0 ns, sys: 239 ms, total: 239 ms
Wall time: 238 ms

pickle - deserializing
CPU times: user 0 ns, sys: 26 ms, total: 26 ms
Wall time: 25.8 ms


Define a remote function which uses the neural net weights.

In [65]:
@ray.remote
def use_weights(weights, i):
    len(weights)
    return i

**EXERCISE:** In the code below, use `ray.put` to avoid copying the neural net weights to the object store multiple times.

In [66]:
time.sleep(2.0)
start_time = time.time()

neural_net_weights_id = ray.put(neural_net_weights)

results = ray.get([use_weights.remote(neural_net_weights_id, i)
                   for i in range(20)])

end_time = time.time()
duration = end_time - start_time

**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [67]:
assert results == list(range(20))
assert duration < 1, ('The experiments ran in {} seconds. This is too '
                      'slow.'.format(duration))

print('Success! The example took {} seconds.'.format(duration))

Success! The example took 0.2656710147857666 seconds.


In [68]:
print('parameter server')
@ray.remote
class ParameterSever:
    def __init__(self):
        self.params = np.zeros(10)

    def get_params(self):
        return self.params

    def update_params(self, grad):
        self.params -= grad


@ray.remote
def worker(ps):
    time.sleep(2)
    grad = np.random.normal(size=10)
    ps.update_params.remote(grad)


param_server = ParameterSever.remote()
print(param_server)

print(f"Initial params: {ray.get(param_server.get_params.remote())}")

print([worker.remote(param_server) for _ in range(3)])

for _ in range(20):
    print(f"Updated params: {ray.get(param_server.get_params.remote())}")
    time.sleep(1)


parameter server
Actor(ParameterSever, 4cbee8d6fda26ea0152d985101000000)
Initial params: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ObjectRef(328cd9d694d92b6affffffffffffffffffffffff0100000001000000), ObjectRef(18b2884533b55af3ffffffffffffffffffffffff0100000001000000), ObjectRef(0b1ae0728cc06aaaffffffffffffffffffffffff0100000001000000)]
Updated params: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Updated params: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Updated params: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Updated params: [ 1.06700538 -0.69678183  0.88154744 -2.42349082  3.95094572  1.00629205
 -0.01872789 -1.16234167  0.91047409  0.70451358]
Updated params: [ 1.06700538 -0.69678183  0.88154744 -2.42349082  3.95094572  1.00629205
 -0.01872789 -1.16234167  0.91047409  0.70451358]
Updated params: [ 1.06700538 -0.69678183  0.88154744 -2.42349082  3.95094572  1.00629205
 -0.01872789 -1.16234167  0.91047409  0.70451358]
Updated params: [ 1.06700538 -0.69678183  0.88154744 -2.42349082  3.95094572  1.00629205
 -0.01872789 -1.16234

## Part 7 - Parallel merge sort (homework)
**Excercise:** Based on the previous examples, create a sequential and parallel implementation of the merge sort algorithm. You can choose any version of the parallel algorithm. Think about optimizations (when to stop spawning new workers). Compare perfromance


In [69]:
def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

def merge_sort_sequential(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort_sequential(arr[:mid])
    right = merge_sort_sequential(arr[mid:])
    return merge(left, right)


PARALLEL_THRESHOLD = 10_000

@ray.remote
def sort_chunk(arr):
    return sorted(arr)

@ray.remote
def merge_chunks(left, right):
    return merge(left, right)

def parallel_merge_sort(arr):
    chunks = []
    for i in range(0, len(arr), PARALLEL_THRESHOLD):
        chunks.append(sort_chunk.remote(arr[i:i + PARALLEL_THRESHOLD]))

    while len(chunks) > 1:
        next_level = []
        for i in range(0, len(chunks), 2):
            if i + 1 < len(chunks):
                next_level.append(merge_chunks.remote(chunks[i], chunks[i + 1]))
            else:
                next_level.append(chunks[i])
        chunks = next_level

    return ray.get(chunks[0])


import random as _random

DATA_SIZE = 200_000
data = [_random.randint(0, 1_000_000) for _ in range(DATA_SIZE)]

t0 = time.time()
sorted_seq = merge_sort_sequential(data)
t_seq = time.time() - t0
print(f'Sequential merge sort: {t_seq:.3f} s')

t0 = time.time()
sorted_par = parallel_merge_sort(data)
t_par = time.time() - t0
print(f'Parallel merge sort:   {t_par:.3f} s')
print(f'Speedup: {t_seq / t_par:.2f}x')

assert sorted_seq == sorted_par, 'Sort results differ!'
print('Results match — sort is correct.')

Sequential merge sort: 0.379 s
Parallel merge sort:   0.105 s
Speedup: 3.62x
Results match — sort is correct.


## Part 8 - Compute Pi using Monte Carlo method (homework)
**Excercise:** Based on the previous example, create an actor-based system that can compute Pi using the Monte Carlo method. There should be one supervising actor and computing tasks/actors. As an output we expect to have numbers approaching to 3.14


![monte-carlo](img/monte_carlo_pi_sma.png)

*Note* I should see something similar to this:

```
Current value of π is: 3.1413560033585224
Current value of π is: 3.1413457351746548
Current value of π is: 3.1413172115005907
Current value of π is: 3.1413644902634594
Current value of π is: 3.1413463306152707
Current value of π is: 3.1412993520518357
Current value of π is: 3.14122792002806
Current value of π is: 3.14125257767156
Current value of π is: 3.1412399868030354
Current value of π is: 3.14132
Current value of π is: 3.1413048336472067
Current value of π is: 3.1412723926380366
Current value of π is: 3.1413103117505994
Current value of π is: 3.141357113523027
Current value of π is: 3.1414312589618585
Current value of π is: 3.141471492704826

In [70]:
@ray.remote
def sample_points(num_samples):
    xs = np.random.uniform(-1, 1, num_samples)
    ys = np.random.uniform(-1, 1, num_samples)
    inside = int(np.sum(xs**2 + ys**2 <= 1.0))
    return inside, num_samples


@ray.remote
class PiSupervisor:
    def __init__(self):
        self.total_inside = 0
        self.total_samples = 0

    def add_result(self, inside, total):
        self.total_inside += inside
        self.total_samples += total

    def get_pi(self):
        if self.total_samples == 0:
            return 0.0
        return 4.0 * self.total_inside / self.total_samples

    def get_stats(self):
        return self.total_inside, self.total_samples


NUM_WORKERS = 8
SAMPLES_PER_WORKER = 1_000_000
NUM_ROUNDS = 16

supervisor = PiSupervisor.remote()

for round_idx in range(NUM_ROUNDS):
    refs = [sample_points.remote(SAMPLES_PER_WORKER) for _ in range(NUM_WORKERS)]

    results = ray.get(refs)
    add_refs = [supervisor.add_result.remote(inside, total) for inside, total in results]
    ray.get(add_refs)

    pi_estimate = ray.get(supervisor.get_pi.remote())
    print(f'Current value of pi is: {pi_estimate}')

Current value of pi is: 3.1412055
Current value of pi is: 3.14095075
Current value of pi is: 3.1412856666666666
Current value of pi is: 3.141323
Current value of pi is: 3.1412678
Current value of pi is: 3.1413460833333335
Current value of pi is: 3.1412530714285714
Current value of pi is: 3.1412775625
Current value of pi is: 3.1414125555555557
Current value of pi is: 3.1414555
Current value of pi is: 3.141439181818182
Current value of pi is: 3.1414803333333334
Current value of pi is: 3.141490923076923
Current value of pi is: 3.1415113928571428
Current value of pi is: 3.141445
Current value of pi is: 3.14147334375


## Clean up  - Clean up the environemnt

**GOAL:** The goal of this exercise is to close the environment once you finish play with ray `ray.shutdown`.

In [71]:
ray.shutdown()